# NutriVision — Fine-tuning Stage 1 (YOLOv11l-seg)

**Tujuan Stage 1:** latih *hanya* neck + segmentation head dengan backbone dibekukan
(`freeze=11`). Ini membangun fondasi stabil sebelum Stage 2 membuka seluruh layer.

**Dataset:** output notebook `data_exploration_repair` — `clean_v1.zip` yang sudah
mengandung gambar sintetis copy-paste dan manifest `train_oversampled.txt`.

**Urutan pipeline:**
```
data_exploration_repair  →  Stage 1 (notebook ini)
                         →  Stage 2 (mulai dari STAGE1_BEST.pt)
                         →  Optuna  (search dari STAGE2_BEST.pt)
                         →  Final evaluation
```

Aktifkan **GPU P100 / T4** di Kaggle sebelum `Run All`.


## 1. Environment

In [ ]:
from pathlib import Path
from collections import Counter
import importlib.util
import json
import os
import random
import shutil
import subprocess
import sys
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import torch
from IPython.display import display, FileLink

# Install Ultralytics jika belum ada (Kaggle kadang sudah bundled)
if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])

try:
    import cv2; cv2.setNumThreads(0)
except Exception:
    pass

from ultralytics import YOLO

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
INPUT_ROOT  = Path("/kaggle/input")
WORK_ROOT   = Path("/kaggle/working")

# Slug dataset yang di-attach dari output data_exploration_repair.
# Di Kaggle: Add Input → Your Work → pilih notebook data_exploration_repair → Output
DATASET_SLUG = "nutrivision-pro-yolo-cleanv1"

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME = "yolo11l-seg.pt"   # pretrained COCO

# ── Training knobs ────────────────────────────────────────────────────────────
IMG_SIZE       = 640
STAGE1_EPOCHS  = 60      # cukup untuk frozen head; early stopping mengamankan
FREEZE_LAYERS  = 11      # bekukan backbone (0–10), latih neck + head
RESUME         = False   # set True untuk lanjutkan run yang terputus

# ── Oversampling flag ─────────────────────────────────────────────────────────
# True  → gunakan train_oversampled.txt  (kelas minoritas muncul 4× lebih sering)
# False → gunakan train/images biasa
USE_OVERSAMPLED_MANIFEST = True

# ── Directories ───────────────────────────────────────────────────────────────
RUNS_ROOT       = WORK_ROOT / "stage1_runs"
EXPERIMENT_NAME = "yolo11l_seg_stage1"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

# ── GPU check ─────────────────────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not detected. Kaggle: Settings > Accelerator > GPU T4 / P100, then Restart & Run All."
    )

DEVICE        = 0
GPU_NAME      = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
WORKERS       = min(4, max(2, (os.cpu_count() or 4) // 2))

print(f"PyTorch  : {torch.__version__}")
print(f"GPU      : {GPU_NAME}")
print(f"VRAM     : {GPU_MEMORY_GB:.1f} GB")
print(f"Workers  : {WORKERS}")
print(f"Model    : {MODEL_NAME}")
print(f"Oversampled manifest: {USE_OVERSAMPLED_MANIFEST}")


## 2. Dataset discovery

Notebook ini mendeteksi dataset secara otomatis dari tiga sumber:
1. Dataset Kaggle yang di-attach langsung (`/kaggle/input/<slug>/`)
2. ZIP `clean_v1.zip` di dalam dataset tersebut (akan di-extract ke `/kaggle/working/`)
3. Folder `clean_v1/` yang sudah ter-extract

Setelah ditemukan, notebook memilih antara `data.yaml` (training standar)
dan `data_oversampled.yaml` (manifest oversampling) sesuai flag `USE_OVERSAMPLED_MANIFEST`.


In [ ]:
def find_clean_dataset_root():
    """Cari root folder clean_v1 di semua kemungkinan lokasi Kaggle."""
    # Prioritas 1: slug langsung
    direct = INPUT_ROOT / DATASET_SLUG
    if direct.exists():
        return direct

    # Prioritas 2: iterasi semua input, cari yang punya data.yaml + splits
    for root in sorted(INPUT_ROOT.iterdir()):
        if not root.is_dir():
            continue
        for yaml_path in root.rglob("data.yaml"):
            parent = yaml_path.parent
            if all((parent / s).is_dir() for s in ("train", "valid", "test")):
                return root

    raise FileNotFoundError(
        f"Dataset tidak ditemukan. Input yang tersedia: "
        f"{[str(p) for p in INPUT_ROOT.iterdir() if p.is_dir()]}"
    )


def choose_yaml(candidates, prefer_oversampled: bool):
    """Pilih data.yaml atau data_oversampled.yaml dari daftar kandidat."""
    candidates = [Path(p) for p in candidates if Path(p).is_file()]
    if not candidates:
        raise FileNotFoundError("Tidak menemukan file YAML dataset.")

    if prefer_oversampled:
        oversampled = [p for p in candidates if "oversampled" in p.name]
        if oversampled:
            return sorted(oversampled, key=lambda p: len(str(p)))[0]
        print("[WARN] data_oversampled.yaml tidak ditemukan — fallback ke data.yaml.")

    standard = [p for p in candidates if p.name == "data.yaml"]
    if standard:
        return sorted(standard, key=lambda p: len(str(p)))[0]

    return sorted(candidates, key=lambda p: len(str(p)))[0]


dataset_input = find_clean_dataset_root()
print("Dataset input :", dataset_input)

# ── Extract ZIP jika perlu ────────────────────────────────────────────────────
zip_candidates = sorted(dataset_input.rglob("clean_v1.zip"))
if zip_candidates:
    extracted_root = WORK_ROOT / "clean_dataset_extracted"
    if not extracted_root.exists():
        extracted_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_candidates[0], "r") as archive:
            archive.extractall(extracted_root)
        print("ZIP extracted:", zip_candidates[0])
    else:
        print("ZIP sudah pernah di-extract, skip.")
    all_yamls = list(extracted_root.rglob("data.yaml")) + list(extracted_root.rglob("data_oversampled.yaml"))
else:
    all_yamls = list(dataset_input.rglob("data.yaml")) + list(dataset_input.rglob("data_oversampled.yaml"))

SOURCE_DATA_YAML = choose_yaml(all_yamls, prefer_oversampled=USE_OVERSAMPLED_MANIFEST)
DATASET_ROOT     = SOURCE_DATA_YAML.parent

with SOURCE_DATA_YAML.open("r", encoding="utf-8") as f:
    DATA_CONFIG = yaml.safe_load(f) or {}

CLASS_NAMES = DATA_CONFIG.get("names", [])
if isinstance(CLASS_NAMES, dict):
    CLASS_NAMES = [CLASS_NAMES[k] for k in sorted(CLASS_NAMES, key=lambda x: int(x))]
CLASS_NAMES = [str(n) for n in CLASS_NAMES]
NUM_CLASSES = int(DATA_CONFIG.get("nc", len(CLASS_NAMES)))
if NUM_CLASSES != len(CLASS_NAMES):
    raise ValueError(f"data.yaml inkonsisten: nc={NUM_CLASSES}, names={len(CLASS_NAMES)}")

print(f"\nSource YAML : {SOURCE_DATA_YAML}")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Classes     : {NUM_CLASSES} → {CLASS_NAMES}")


In [ ]:
# ── Build runtime YAML dengan absolute path ───────────────────────────────────
# Ultralytics membaca path dari YAML saat training. Karena YAML bisa mengandung
# path /kaggle/working dari sesi data_exploration yang berbeda, kita buat ulang
# dengan path absolut sesi ini.

def resolve(value):
    p = Path(str(value))
    return p if p.is_absolute() else DATASET_ROOT / p

def count_files(path, exts):
    if not Path(path).exists():
        return 0
    return sum(1 for p in Path(path).rglob("*") if p.is_file() and p.suffix.lower() in exts)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# train bisa berupa folder atau path ke .txt manifest
train_raw   = DATA_CONFIG.get("train", "train/images")
train_value = resolve(train_raw)

# Jika manifest oversampled, path sudah absolut di dalam YAML — gunakan langsung
# Jika folder, build path normal
if train_value.suffix == ".txt":
    TRAIN_PATH = train_value   # manifest .txt
    train_display = f"manifest: {train_value.name}"
else:
    TRAIN_PATH = train_value   # folder
    train_display = f"folder: {count_files(train_value, IMAGE_EXTS)} images"

VAL_IMAGES  = resolve(DATA_CONFIG.get("val",  "valid/images"))
TEST_IMAGES = resolve(DATA_CONFIG["test"]) if DATA_CONFIG.get("test") else None

RUNTIME_DATA_YAML = WORK_ROOT / "stage1_runtime_data.yaml"
runtime_config = {
    "path":  str(DATASET_ROOT),
    "train": str(TRAIN_PATH),
    "val":   str(VAL_IMAGES),
    "nc":    NUM_CLASSES,
    "names": CLASS_NAMES,
}
if TEST_IMAGES:
    runtime_config["test"] = str(TEST_IMAGES)

RUNTIME_DATA_YAML.write_text(
    yaml.safe_dump(runtime_config, sort_keys=False, allow_unicode=True),
    encoding="utf-8"
)
DATA_YAML = RUNTIME_DATA_YAML

print(f"Train       : {train_display}")
print(f"Val images  : {count_files(VAL_IMAGES, IMAGE_EXTS)}")
if TEST_IMAGES:
    print(f"Test images : {count_files(TEST_IMAGES, IMAGE_EXTS)}")
print(f"Runtime YAML: {DATA_YAML}")
print()
print(DATA_YAML.read_text(encoding="utf-8"))


## 3. Load pretrained model dan inspect arsitektur

In [ ]:
model = YOLO(MODEL_NAME)
model.info(verbose=False)

total_params = sum(p.numel() for p in model.model.parameters())
print(f"Pretrained model : {MODEL_NAME}")
print(f"Total parameters : {total_params:,}")
print(f"Layers           : {len(model.model.model)}")
print()
print("Layer index | Class                    | Params")
print("-" * 55)
for idx, layer in enumerate(model.model.model):
    p = sum(x.numel() for x in layer.parameters())
    frozen_marker = " ← FROZEN" if idx < FREEZE_LAYERS else ""
    print(f"  {idx:02d}        | {layer.__class__.__name__:<24s} | {p:>10,}{frozen_marker}")


## 4. Stage 1 training configuration

### Filosofi config ini

| Parameter | Nilai | Alasan |
|---|---|---|
| `freeze=11` | backbone frozen | head masih random init dari head COCO → perlu LR tinggi |
| `lr0=0.001` | tinggi relatif | wajar untuk cold-start head |
| `lrf=0.01` | final LR = lr0×lrf = 1e-5 | turun cukup jauh agar konverge |
| `warmup_epochs=3` | 3 epoch | butuh warmup karena cold start |
| `cos_lr=True` | cosine decay | lebih smooth daripada linear |
| `mosaic=1.0` | penuh | dataset kecil → mosaic sangat membantu generalisasi |
| `copy_paste=0.30` | agresif | kelas minoritas butuh exposure extra di level augmentasi online |
| `copy_paste_mode="flip"` | flip | lebih conservative dari random, aman untuk makanan |
| `mixup=0.05` | kecil | sedikit saja, jangan terlalu banyak distorsi di tahap head |
| `close_mosaic=10` | 10 epoch terakhir | biarkan model stabilize tanpa mosaic di akhir |
| `patience=20` | 20 epoch | dataset kecil, plateau bisa lama |
| `amp=True` | mixed precision | VRAM efficiency di T4/P100 |
| `overlap_mask=True` | | handle instance yang overlap di piring |
| `mask_ratio=4` | | resolusi mask lebih halus dari default (4 vs 4) |
| `batch=-1` | auto | Ultralytics pilih batch size optimal untuk VRAM |


In [ ]:
STAGE1_CONFIG = {
    # ── Dataset & hardware ────────────────────────────────────────────────────
    "data":          str(DATA_YAML),
    "epochs":        STAGE1_EPOCHS,
    "imgsz":         IMG_SIZE,
    "batch":         -1,          # auto-batch: Ultralytics pilih sesuai VRAM
    "device":        DEVICE,
    "workers":       WORKERS,
    "freeze":        FREEZE_LAYERS,

    # ── Optimizer ─────────────────────────────────────────────────────────────
    "optimizer":     "AdamW",
    "lr0":           0.001,       # LR awal — tinggi karena cold-start head
    "lrf":           0.01,        # final LR = lr0 × lrf = 1e-5
    "momentum":      0.937,
    "weight_decay":  0.0005,
    "cos_lr":        True,        # cosine annealing
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr":  0.1,

    # ── Regularization & training control ────────────────────────────────────
    "patience":      20,
    "close_mosaic":  10,
    "amp":           True,
    "deterministic": True,
    "seed":          SEED,
    "save":          True,
    "save_period":   5,
    "plots":         True,
    "val":           True,

    # ── Segmentation-specific ─────────────────────────────────────────────────
    "overlap_mask":  True,
    "mask_ratio":    4,

    # ── Augmentation (segmentation-friendly, agresif di mosaic & copy_paste) ──
    "hsv_h":         0.015,   # hue shift ringan
    "hsv_s":         0.70,    # saturation cukup agresif (makanan colorful)
    "hsv_v":         0.40,    # brightness variation
    "degrees":       5.0,     # rotasi kecil (makanan di piring tidak terlalu rotate)
    "translate":     0.10,
    "scale":         0.50,    # scale augmentation cukup besar
    "shear":         2.0,
    "perspective":   0.0005,
    "flipud":        0.0,     # jangan flip vertikal (makanan punya orientasi)
    "fliplr":        0.5,
    "mosaic":        1.0,     # mosaic penuh — paling efektif untuk dataset kecil
    "mixup":         0.05,    # kecil saja di stage 1
    "copy_paste":    0.30,    # agresif — bantu kelas minoritas
    "copy_paste_mode": "flip",
}

print("Stage 1 Config:")
print(json.dumps(STAGE1_CONFIG, indent=2, default=str))


## 5. Run training Stage 1

In [ ]:
EXPERIMENT_DIR  = RUNS_ROOT / EXPERIMENT_NAME
LAST_CHECKPOINT = EXPERIMENT_DIR / "weights" / "last.pt"

if RESUME and LAST_CHECKPOINT.exists():
    print("Resuming from:", LAST_CHECKPOINT)
    train_model    = YOLO(str(LAST_CHECKPOINT))
    resume_arg     = True
else:
    train_model = model
    resume_arg  = False

train_results = train_model.train(
    **STAGE1_CONFIG,
    project  = str(RUNS_ROOT),
    name     = EXPERIMENT_NAME,
    exist_ok = True,
    resume   = resume_arg,
    verbose  = True,
)

RUN_DIR          = Path(train_model.trainer.save_dir)
BEST_MODEL_PATH  = RUN_DIR / "weights" / "best.pt"
LAST_MODEL_PATH  = RUN_DIR / "weights" / "last.pt"

print("\n" + "=" * 72)
print("STAGE 1 TRAINING DONE")
print("=" * 72)
print("Run dir  :", RUN_DIR)
print("Best     :", BEST_MODEL_PATH)
print("Last     :", LAST_MODEL_PATH)
assert BEST_MODEL_PATH.exists(), f"best.pt tidak ditemukan: {BEST_MODEL_PATH}"


## 6. Training history

In [ ]:
best_model  = YOLO(str(BEST_MODEL_PATH))
results_csv = RUN_DIR / "results.csv"

if results_csv.exists():
    history_df = pd.read_csv(results_csv)
    history_df.columns = [c.strip() for c in history_df.columns]
    display(history_df.tail(10))

    loss_cols = [c for c in history_df.columns if "loss" in c.lower()]
    map_cols  = [c for c in history_df.columns if "map" in c.lower()]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for col in loss_cols:
        axes[0].plot(history_df["epoch"], history_df[col], label=col)
    axes[0].set_title("Loss curves"); axes[0].legend(fontsize=8); axes[0].grid(True)

    for col in map_cols:
        axes[1].plot(history_df["epoch"], history_df[col], label=col)
    axes[1].set_title("mAP curves"); axes[1].legend(fontsize=8); axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(RUN_DIR / "stage1_training_curves.png", dpi=100)
    plt.show()
else:
    print("results.csv tidak ditemukan:", results_csv)


## 7. Evaluation — val dan test split

In [ ]:
EVAL_ROOT = RUN_DIR / "evaluation"
EVAL_ROOT.mkdir(parents=True, exist_ok=True)

# Evaluasi selalu menggunakan data.yaml standar (bukan manifest oversampled)
# agar metrik tidak bias oleh repetisi gambar minority
EVAL_DATA_YAML = WORK_ROOT / "stage1_eval_data.yaml"
eval_config = {
    "path":  str(DATASET_ROOT),
    "train": str(DATASET_ROOT / "train" / "images"),
    "val":   str(VAL_IMAGES),
    "nc":    NUM_CLASSES,
    "names": CLASS_NAMES,
}
if TEST_IMAGES:
    eval_config["test"] = str(TEST_IMAGES)
EVAL_DATA_YAML.write_text(
    yaml.safe_dump(eval_config, sort_keys=False, allow_unicode=True),
    encoding="utf-8"
)

def flatten_results(results, split_name):
    row = {"split": split_name}
    for key, val in (getattr(results, "results_dict", {}) or {}).items():
        try:    row[key] = float(val)
        except: row[key] = str(val)
    return row

evaluation_rows    = []
evaluation_results = {}

for split_name in ("val", "test"):
    if split_name == "test" and not TEST_IMAGES:
        continue
    print(f"\nEvaluating split={split_name} ...")
    split_results = best_model.val(
        data    = str(EVAL_DATA_YAML),
        split   = split_name,
        imgsz   = IMG_SIZE,
        batch   = 4,
        device  = DEVICE,
        workers = WORKERS,
        plots   = True,
        project = str(EVAL_ROOT),
        name    = split_name,
        exist_ok= True,
        verbose = True,
    )
    evaluation_results[split_name] = split_results
    evaluation_rows.append(flatten_results(split_results, split_name))

evaluation_df = pd.DataFrame(evaluation_rows)
display(evaluation_df)
evaluation_df.to_csv(EVAL_ROOT / "stage1_split_metrics.csv", index=False)


## 8. Per-class segmentation metrics

In [ ]:
def per_class_seg_metrics(results, class_names):
    seg = getattr(results, "seg", None)
    if seg is None:
        return pd.DataFrame()
    indices = getattr(seg, "ap_class_index", None)
    if indices is None:
        return pd.DataFrame()
    indices = [int(i) for i in np.asarray(indices).tolist()]
    p_arr   = np.asarray(getattr(seg, "p",    []))
    r_arr   = np.asarray(getattr(seg, "r",    []))
    ap50    = np.asarray(getattr(seg, "ap50", []))
    ap      = np.asarray(getattr(seg, "ap",   []))
    rows = []
    for pos, cls_id in enumerate(indices):
        row = {
            "class_id":   cls_id,
            "class_name": class_names[cls_id] if cls_id < len(class_names) else str(cls_id),
        }
        for name, arr in [("precision", p_arr), ("recall", r_arr),
                           ("mAP50", ap50), ("mAP50-95", ap)]:
            if pos < len(arr):
                row[name] = float(arr[pos])
        if "precision" in row and "recall" in row:
            denom = row["precision"] + row["recall"]
            row["f1"] = 2 * row["precision"] * row["recall"] / denom if denom > 0 else 0.0
        rows.append(row)
    return pd.DataFrame(rows)

all_per_class = []
for split_name, split_results in evaluation_results.items():
    tbl = per_class_seg_metrics(split_results, CLASS_NAMES)
    if not tbl.empty:
        tbl.insert(0, "split", split_name)
        all_per_class.append(tbl)

if all_per_class:
    per_class_df = pd.concat(all_per_class, ignore_index=True).round(4)
    display(per_class_df.sort_values(["split", "mAP50-95"]))
    per_class_df.to_csv(EVAL_ROOT / "stage1_per_class_metrics.csv", index=False)

    # Plot recall per class — metrik utama NutriVision
    test_tbl = per_class_df[per_class_df["split"] == "test"].sort_values("recall")
    if not test_tbl.empty:
        fig, ax = plt.subplots(figsize=(12, 5))
        colors = ["#e34948" if v < 0.5 else "#f5a623" if v < 0.7 else "#1baf7a"
                  for v in test_tbl["recall"]]
        ax.barh(test_tbl["class_name"], test_tbl["recall"], color=colors)
        ax.axvline(0.5, color="red",    linewidth=1, linestyle="--", label="recall=0.50")
        ax.axvline(0.7, color="orange", linewidth=1, linestyle="--", label="recall=0.70")
        ax.set_xlabel("Recall"); ax.set_title("Stage 1 — Per-class Recall (test set)")
        ax.legend(); plt.tight_layout()
        plt.savefig(EVAL_ROOT / "stage1_recall_per_class.png", dpi=100)
        plt.show()
else:
    print("Per-class metrics tidak tersedia (versi Ultralytics terlalu lama).")


## 9. Save artifact Stage 1

In [ ]:
# ── Prediction preview ────────────────────────────────────────────────────────
PRED_DIR = EVAL_ROOT / "test_predictions"
if TEST_IMAGES:
    img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    test_imgs = sorted(p for p in TEST_IMAGES.rglob("*")
                       if p.is_file() and p.suffix.lower() in img_exts)
    previews = test_imgs[:min(24, len(test_imgs))]
    if previews:
        best_model.predict(
            source  = [str(p) for p in previews],
            conf    = 0.25,
            imgsz   = IMG_SIZE,
            device  = DEVICE,
            save    = True,
            project = str(PRED_DIR.parent),
            name    = PRED_DIR.name,
            exist_ok= True,
            verbose = False,
        )
        print(f"Preview saved to: {PRED_DIR}")

# ── Summary JSON ──────────────────────────────────────────────────────────────
SUMMARY_JSON = EVAL_ROOT / "stage1_summary.json"
summary = {
    "stage":              "stage1",
    "model_base":         MODEL_NAME,
    "dataset_slug":       DATASET_SLUG,
    "use_oversampled":    USE_OVERSAMPLED_MANIFEST,
    "source_data_yaml":   str(SOURCE_DATA_YAML),
    "runtime_data_yaml":  str(DATA_YAML),
    "run_dir":            str(RUN_DIR),
    "best_model":         str(BEST_MODEL_PATH),
    "last_model":         str(LAST_MODEL_PATH),
    "imgsz":              IMG_SIZE,
    "epochs_requested":   STAGE1_EPOCHS,
    "freeze_layers":      FREEZE_LAYERS,
    "stage1_config":      {k: str(v) if isinstance(v, Path) else v
                           for k, v in STAGE1_CONFIG.items()},
    "gpu":                GPU_NAME,
    "gpu_memory_gb":      round(GPU_MEMORY_GB, 2),
}
SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")

# ── ZIP artifacts ─────────────────────────────────────────────────────────────
ARTIFACT_ZIP = WORK_ROOT / "stage1_artifacts.zip"
if ARTIFACT_ZIP.exists():
    ARTIFACT_ZIP.unlink()
with zipfile.ZipFile(ARTIFACT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in RUN_DIR.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(RUN_DIR.parent))

print("=" * 72)
print("STAGE 1 COMPLETE")
print("=" * 72)
print("Run dir       :", RUN_DIR)
print("Best model    :", BEST_MODEL_PATH)
print("Artifact ZIP  :", ARTIFACT_ZIP)
print("\nLangkah selanjutnya:")
print("  1. Commit output notebook ini (Save Version → Save & Run All)")
print("  2. Di notebook Stage 2: Add Input → Your Work → notebook ini → Output")
print("  3. Stage 2 akan otomatis menemukan BEST_MODEL_PATH sebagai starting weight")


## 10. Download links

In [ ]:
display(FileLink(str(ARTIFACT_ZIP)))
display(FileLink(str(BEST_MODEL_PATH)))
display(FileLink(str(EVAL_ROOT / "stage1_split_metrics.csv")))
display(FileLink(str(EVAL_ROOT / "stage1_per_class_metrics.csv") if (EVAL_ROOT / "stage1_per_class_metrics.csv").exists() else str(SUMMARY_JSON)))
display(FileLink(str(SUMMARY_JSON)))


---
## Checklist sebelum Stage 2

- [ ] Notebook ini sudah di-commit (Save Version → Save & Run All, bukan Quick Save)
- [ ] `best.pt` muncul di Output tab notebook ini
- [ ] Metrics test set: mask recall > 0.30 dan mAP50-95 > 0.20 (baseline minimal)
- [ ] Tidak ada kelas dengan recall = 0.00 di test set
- [ ] Di notebook Stage 2: Add Input → Your Work → notebook Stage 1 ini

> **Catatan YAML evaluasi:** notebook ini menggunakan `stage1_eval_data.yaml`
> (bukan manifest oversampled) untuk evaluasi, sehingga metrik tidak bias oleh
> repetisi gambar kelas minoritas. Angka yang kamu lihat di atas adalah performa
> *real* pada distribusi data yang seimbang.
